#### ABLATION STUDY: Quantify Off-Chain Contribution

In [1]:
import pandas as pd

df = pd.read_parquet("../data/processed/final/final_train_data.parquet")

In [2]:
from sklearn.model_selection import train_test_split

addresses = df['address'].unique()
train_addr, test_addr = train_test_split(addresses, test_size=0.2, random_state=42)

train_df = df[df['address'].isin(train_addr)]
test_df = df[df['address'].isin(test_addr)]

In [3]:
import xgboost as xgb
from sklearn.metrics import f1_score, roc_auc_score

on_chain_features = [
    'normal_total_cnt', 'uniq_peers_cnt',
    'burst_max_tx_5m', 'normal_sent_cnt'
]

market_features = [
    'eth_volatility_7d',
    'eth_daily_return',
    'eth_intraday_volatility'
]

reddit_features = [
    'reddit_fraud_mention_ratio',
    'reddit_total_activity',
    'reddit_avg_sentiment'
]

# Train and evaluate different feature combinations
results = []

# Model 1: On-chain only (baseline)
model_onchain = xgb.XGBClassifier(
    n_estimators=100, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, random_state=42
)
model_onchain.fit(train_df[on_chain_features], train_df['is_anomalous'])
y_pred = model_onchain.predict(test_df[on_chain_features])
y_prob = model_onchain.predict_proba(test_df[on_chain_features])[:,1]

results.append({
    'Model': 'On-Chain Only',
    'Features': 'On-Chain (4)',
    'F1-Score': f1_score(test_df['is_anomalous'], y_pred),
    'ROC-AUC': roc_auc_score(test_df['is_anomalous'], y_prob)
})

# Model 2: On-chain + Market
model_market = xgb.XGBClassifier(
    n_estimators=100, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, random_state=42
)
features_market = on_chain_features + market_features
model_market.fit(train_df[features_market], train_df['is_anomalous'])
y_pred = model_market.predict(test_df[features_market])
y_prob = model_market.predict_proba(test_df[features_market])[:,1]

results.append({
    'Model': 'On-Chain + Market',
    'Features': 'On-Chain (4) + Market (3)',
    'F1-Score': f1_score(test_df['is_anomalous'], y_pred),
    'ROC-AUC': roc_auc_score(test_df['is_anomalous'], y_prob)
})

# Model 3: On-chain + Reddit
model_reddit = xgb.XGBClassifier(
    n_estimators=100, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, random_state=42
)
features_reddit = on_chain_features + reddit_features
model_reddit.fit(train_df[features_reddit], train_df['is_anomalous'])
y_pred = model_reddit.predict(test_df[features_reddit])
y_prob = model_reddit.predict_proba(test_df[features_reddit])[:,1]

results.append({
    'Model': 'On-Chain + Reddit',
    'Features': 'On-Chain (4) + Reddit (3)',
    'F1-Score': f1_score(test_df['is_anomalous'], y_pred),
    'ROC-AUC': roc_auc_score(test_df['is_anomalous'], y_prob)
})

# Model 4: Full (all features)
model_full = xgb.XGBClassifier(
    n_estimators=100, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, random_state=42
)
features_full = on_chain_features + market_features + reddit_features
model_full.fit(train_df[features_full], train_df['is_anomalous'])
y_pred = model_full.predict(test_df[features_full])
y_prob = model_full.predict_proba(test_df[features_full])[:,1]

results.append({
    'Model': 'Full Multimodal',
    'Features': 'On-Chain (4) + Market (3) + Reddit (3)',
    'F1-Score': f1_score(test_df['is_anomalous'], y_pred),
    'ROC-AUC': roc_auc_score(test_df['is_anomalous'], y_prob)
})

ablation_df = pd.DataFrame(results)
ablation_df['F1 Improvement'] = ablation_df['F1-Score'] - ablation_df.iloc[0]['F1-Score']
ablation_df['AUC Improvement'] = ablation_df['ROC-AUC'] - ablation_df.iloc[0]['ROC-AUC']

print(ablation_df.to_string(index=False))

            Model                               Features  F1-Score  ROC-AUC  F1 Improvement  AUC Improvement
    On-Chain Only                           On-Chain (4)  0.748907 0.825608        0.000000         0.000000
On-Chain + Market              On-Chain (4) + Market (3)  0.747927 0.824406       -0.000980        -0.001202
On-Chain + Reddit              On-Chain (4) + Reddit (3)  0.750282 0.826071        0.001375         0.000463
  Full Multimodal On-Chain (4) + Market (3) + Reddit (3)  0.750431 0.825270        0.001525        -0.000339
